# Job Description Parser

**NLP IE — Extract skills, experience, location, salary from job postings**

## Project Overview

Parse job descriptions to extract **skills**, **experience years**,
**location**, **salary range**, and **degree requirements** using regex and spaCy NER.

## Learning Objectives

1. Use regex + NER for JD parsing.
2. Handle varied text formats.
3. Build a reusable parser.

## Problem Statement

Extract structured fields from raw job posting text.

## Why This Project Matters

- Millions of JDs processed daily.
- Structured data enables job-candidate matching.
- Skills extraction reveals market trends.

## Dataset Overview

8 synthetic JDs.

## Dataset Source and License Notes

Synthetic.

## Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["pandas","numpy","matplotlib","seaborn","spacy"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable,"-m","spacy","download","en_core_web_sm"])
print("Environment ready.")


## Imports

In [ ]:
import re, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import spacy
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline
nlp = spacy.load("en_core_web_sm")
SEED = 42; np.random.seed(SEED)
print("Loaded.")


## Configuration / Constants

In [ ]:
SKILLS_RE = r"\b(?:Python|Java|JavaScript|TypeScript|SQL|R|C\+\+|Go|Docker|AWS|GCP|Azure|PyTorch|TensorFlow|Django|React|Kubernetes|Terraform|Spark|Tableau|Git|CI/CD|Linux|PostgreSQL|MongoDB|Flask|FastAPI|Spring)\b"
print("Skills pattern configured.")


## Dataset Download or Loading

In [ ]:
JDS = [
    "Senior Python Developer — Remote. 5+ years Python, Django, AWS. $120K-$150K. BS required.",
    "ML Engineer in San Francisco. 3+ years ML, PyTorch, TensorFlow. MS preferred.",
    "Data Scientist (NYC) — 2-4 years. Python, R, SQL, Tableau. $90K-$110K. Bachelor in CS.",
    "DevOps Engineer — Austin, TX. 4+ years. Docker, Kubernetes, Terraform, AWS, GCP, Linux. $130K-$160K.",
    "Full Stack Developer (Remote). JavaScript, TypeScript, React, Python, PostgreSQL. 3+ years. $100K-$130K. BS.",
    "Senior Data Engineer — Seattle. 6+ years. Python, Spark, AWS, Kubernetes, SQL. $140K-$170K. MS preferred.",
    "Backend Developer in Chicago. 2+ years Java, Spring, PostgreSQL, Docker, Git. $85K-$105K.",
    "AI Research Scientist — Boston. PhD required. PyTorch, TensorFlow, Python, C++. 3+ years. $150K-$200K.",
]
df = pd.DataFrame({"text": JDS})
print(f"Loaded {len(df)} JDs")


## Data Validation Checks

In [ ]:
assert df["text"].notna().all()
print(f"Validated: {len(df)} JDs")


## Exploratory Data Analysis

In [ ]:
df["words"] = df["text"].str.split().apply(len)
print(f"Avg words: {df["words"].mean():.0f}")


## Target Analysis

Targets: skills, experience, location, salary, degree.

## Train/Validation/Test Split Strategy

Not applicable — rule-based.

## Preprocessing Strategy

Regex for skills/experience/salary/degree. spaCy NER for location.

## Feature Engineering — JD Parser

In [ ]:
def parse_jd(text):
    result = {}
    m = re.search(r"(\d+)\+?\s*(?:years?|yrs?)", text, re.IGNORECASE)
    result["min_exp"] = int(m.group(1)) if m else None
    sals = re.findall(r"\$(\d{2,3})K", text)
    result["salary"] = [int(s)*1000 for s in sals] if sals else None
    doc = nlp(text)
    locs = [e.text for e in doc.ents if e.label_ == "GPE"]
    result["location"] = locs[0] if locs else ("Remote" if "remote" in text.lower() else None)
    result["skills"] = sorted(set(re.findall(SKILLS_RE, text, re.IGNORECASE)))
    degs = re.findall(r"\b(?:Bachelor|Master|MS|BS|PhD|MBA)\b", text, re.IGNORECASE)
    result["degree"] = list(set(degs))
    return result
print("parse_jd() ready.")


## Baseline Model — Parse All JDs

In [ ]:
results = []
for i, jd in enumerate(JDS):
    p = parse_jd(jd)
    results.append(p)
    print(f"\nJD {i+1}: {jd[:40]}...")
    for k,v in p.items(): print(f"  {k}: {v}")


## LazyPredict Benchmark

> **Not applicable.** NLP IE task.

## FLAML AutoML Run

> **Not applicable.** FLAML not used for NLP IE.

## Additional Best-Library Workflow — Skills Frequency

In [ ]:
from collections import Counter
all_skills = [s for r in results for s in r["skills"]]
top = Counter(all_skills).most_common(15)
fig, ax = plt.subplots(figsize=(10,5))
ax.barh([s for s,_ in top], [c for _,c in top], color="steelblue")
ax.set_title("Most Demanded Skills"); ax.invert_yaxis()
plt.tight_layout(); plt.show()


## Top 3 Model Selection

Single extraction pipeline.

## Final Training and Evaluation of Top 3

Regex + NER pipeline is the final system.

## Error Analysis

In [ ]:
for i, r in enumerate(results):
    m = [k for k,v in r.items() if v is None or v == []]
    if m: print(f"JD {i+1}: missing {m}")


## Interpretation and Business Insight

1. Python, SQL, Docker most demanded.
2. Regex effectively extracts salary/experience.
3. spaCy NER catches locations.
4. Real JDs vary more.

## Limitations

- Regex fragile across formats.
- Limited salary format.
- English-only.

## How to Improve This Project

1. Custom NER on labeled JD data.
2. GLiNER for zero-shot skills.
3. Multiple salary formats.

## Production Considerations

- Handle diverse JD formats.
- Update skill dictionaries.
- Add confidence scores.

## Common Mistakes

1. Hardcoding salary format.
2. Incomplete skills list.
3. Not handling Remote/Hybrid.

## Mini Challenge / Exercises

1. Add benefit extraction.
2. Handle multiple currencies.
3. Build JD similarity scorer.

## Final Summary / Key Takeaways

1. Regex + spaCy NER parses structured JDs.
2. Skills dictionaries need maintenance.
3. JD parsing is essential for HR tech.
4. For production, combine rules with trained models.